### README

From coord, get features needed for ML prediction.

### Requirements

In [ ]:
# Python libraries
"""
biopython==1.85
pandas==2.2.3
"""

# Outer softwares
"""
SVM-BPfinder-3M: Corvelo A, Hallegger M, Smith CW, Eyras E. Genome-wide association between branch point properties and alternative splicing. PLoS Comput Biol. 2010 Nov 24;6(11):e1001016. doi: 10.1371/journal.pcbi.1001016. PMID: 21124863; PMCID: PMC2991248.
maxentscan: Yeo G, Burge CB. Maximum entropy modeling of short sequence motifs with applications to RNA splicing signals. J Comput Biol. 2004;11(2-3):377-94. doi: 10.1089/1066527041410418. PMID: 15285897.
"""

'\npandas==2.2.3\nbiopython==1.85\n'

In [ ]:
# Import libraries
import pandas as pd
import subprocess
import io
import tempfile
import os


# Outer softwares
bpfinder = '/PATH/TO/SVM-BPfinder-3M/svm_bpfinder.py'
maxentscan = '/PATH/TO/maxEntScan/fordownload'

### Constants

In [ ]:
# Input: path to file with sequence of minigenes
SEQ_PATH = '/PATH/TO/Minigene_list.xlsx' 
INTRON_COL = 'intron'
DNEXON_COL = 'downstream exon'

# Output: set directory
OUTPUT_DIR = '/PATH/TO/OUTPUT'
OUTPUT_PREFIX = 'minigene'


### Functions

In [4]:
# CALCULATE SPLICE SITE SCORES

def maxentscan_score_3ss(sequences):
    
    # Create a temp file for input sequences
    with tempfile.NamedTemporaryFile(mode='w+', delete=False) as temp_input:
        for seq in sequences:
            temp_input.write(seq + '\n')
        temp_input_name = temp_input.name

    try:
        script_path = os.path.join(maxentscan, 'score3.pl')

        # Run score3.pl from maxentscan_dir to ensure matrix files found
        result = subprocess.run(
            ['perl', script_path, temp_input_name],
            cwd=maxentscan,
            capture_output=True,
            text=True,
            check=True
        )

        # Parse scores: each line is "<sequence>\t<score>"
        scores = [float(line.strip().split('\t')[1]) for line in result.stdout.strip().split('\n')]

    finally:
        # Clean up temp file
        os.remove(temp_input_name)

    return scores

In [5]:
# CALCULATE BRANCH POINT SCORES

# writes a fasta file with intron sequece, each intron is an identifier
def write_fasta(seq_dict, fasta_filename, trim=(0,0)):
    # trim: number of bases do remove from 5' and 3' end
    start_trim, end_trim = trim

    with open(fasta_filename, "w") as fasta_file:
        for intron_id, seq in seq_dict.items():
            seq = str(seq)
            
            # ignore seq shorter that trim
            if len(seq) <= start_trim + end_trim:
                continue
            
            trimmed_seq = seq[start_trim:-end_trim] if end_trim > 0 else seq[start_trim:]
            fasta_file.write(f">{intron_id}\n{trimmed_seq}\n")

# get BPfinder output from fasta file
# output is a dataframe showing only the best prediction of BP per intron
def find_bp(fasta_filename):
    
    # Run SVM-BPfinder
    cmd = f"{bpfinder} -i {fasta_filename} -s Hsap -l 100 -d 10"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    # Check for errors
    if result.returncode != 0:
        print("Error running SVM-BPfinder:", result.stderr)
        return None

    # Convert output into a DataFrame
    svm_output = pd.read_csv(io.StringIO(result.stdout), sep="\t")
    
    # Ensure `svm_scr` is numeric
    svm_output["svm_scr"] = pd.to_numeric(svm_output["svm_scr"], errors="coerce")

    # Select the row with the highest `svm_scr` for each intron
    svm_output_best = svm_output.loc[svm_output.groupby("seq_id")["svm_scr"].idxmax()]
    
    return svm_output_best

### Analysis

##### 1. Read input files

In [ ]:
# sequence table
data_df = pd.read_csv(SEQ_PATH)

# select only NAGNAG
data_df = data_df[data_df['NNAGNAG type'] != 'no NAGNAG'].copy()
data_df

,name,backbone,sequence complete,upstream exon,intron,downstream exon,NNAGNAG type,PSI ctrl. (s1),PSI ctrl. (s2),PSI ctrl. (s3),...,PSI ctrl. (s6),PSI FAM32AD17 (s1),PSI FAM32AD17 (s2),PSI FAM32AD17 (s3),PSI FAM32AD17 (s4),PSI FAM32AD17 (s5),PSI FAM32AD17 (s6),mean delta PSI,regulated (true/false) dPSI>15%,position mutation (AG gain)
0,ppp1r12c CAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CCAGCAG,64.04,68.57,65.51,...,NaN,27.96,28.83,27.89,NaN,NaN,NaN,37.81,True,na
1,gpank1 CAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CCAGCAG,72.31,73.20,71.15,...,NaN,36.70,36.92,36.87,NaN,NaN,NaN,35.39,True,na
2,acin1 CAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,ACAGCAG,65.57,71.78,67.65,...,NaN,54.66,55.62,57.41,NaN,NaN,NaN,12.44,False,na
3,ankrd11 AAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,GAAGCAG,66.78,66.22,68.59,...,NaN,53.71,47.53,51.83,NaN,NaN,NaN,16.17,True,na
4,ppp1r12c CAGCAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CCAGCAG,65.04,62.39,55.11,...,NaN,16.41,12.03,19.85,16.14,NaN,NaN,36.27,True,na
5,ppp1r12c UAGUAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CTAGTAG,59.14,61.24,57.41,...,NaN,26.18,21.61,21.38,24.34,NaN,NaN,38.02,True,na
6,ppp1r12c CAGAAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CCAGAAG,86.88,83.07,81.34,...,NaN,69.29,70.66,67.23,66.39,NaN,NaN,13.66,False,na
7,ppp1r12c AAGCAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CAAGCAG,19.60,23.06,14.17,...,NaN,2.77,0.00,1.35,1.47,NaN,NaN,17.43,True,na
9,ppp1r12c UAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CTAGCAG,65.74,69.00,NaN,...,NaN,32.25,42.45,NaN,NaN,NaN,NaN,30.02,True,na
10,ppp1r12c CAGUAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CCAGTAG,80.91,78.51,NaN,...,NaN,53.68,61.64,NaN,NaN,NaN,NaN,22.05,True,na


##### 2. Get features: 3' end sequence

In [ ]:
# dictionary with intron seq + 5nt dw exon
seq_dict = (
    data_df[INTRON_COL] + data_df[DNEXON_COL].str[:5]
).to_dict()



# add seq information to the dataframe
for i in range(-15,-8):  # intronic part of the 3' end
    col_name = f"3end_{i+8}_base"  
    data_df[col_name] = data_df.index.map(
        lambda idx: str(seq_dict[idx])[i] if idx in seq_dict and len(str(seq_dict[idx])) > i else None
    )

for i in range(-8,0):  # exonic part of the 3' end
    col_name = f"3end_{i+9}_base"  
    data_df[col_name] = data_df.index.map(
        lambda idx: str(seq_dict[idx])[i] if idx in seq_dict and len(str(seq_dict[idx])) > i else None
    )

data_df

,name,backbone,sequence complete,upstream exon,intron,downstream exon,NNAGNAG type,PSI ctrl. (s1),PSI ctrl. (s2),PSI ctrl. (s3),...,3end_-2_base,3end_-1_base,3end_1_base,3end_2_base,3end_3_base,3end_4_base,3end_5_base,3end_6_base,3end_7_base,3end_8_base
0,ppp1r12c CAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CCAGCAG,64.04,68.57,65.51,...,a,g,c,a,g,G,A,G,C,G
1,gpank1 CAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CCAGCAG,72.31,73.20,71.15,...,a,g,c,a,g,G,A,G,C,G
2,acin1 CAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,ACAGCAG,65.57,71.78,67.65,...,a,g,c,a,g,G,A,G,C,G
3,ankrd11 AAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,GAAGCAG,66.78,66.22,68.59,...,a,g,c,a,g,G,A,G,C,G
4,ppp1r12c CAGCAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CCAGCAG,65.04,62.39,55.11,...,a,g,c,a,g,C,T,T,G,A
5,ppp1r12c UAGUAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CTAGTAG,59.14,61.24,57.41,...,a,g,u,a,g,C,T,T,G,A
6,ppp1r12c CAGAAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CCAGAAG,86.88,83.07,81.34,...,a,g,a,a,g,C,T,T,G,A
7,ppp1r12c AAGCAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CAAGCAG,19.60,23.06,14.17,...,a,g,c,a,g,C,T,T,G,A
9,ppp1r12c UAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CTAGCAG,65.74,69.00,NaN,...,a,g,c,a,g,G,A,G,C,G
10,ppp1r12c CAGUAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CCAGTAG,80.91,78.51,NaN,...,a,g,t,a,g,G,A,G,C,G


##### 3. Get features: splice site scores

In [9]:
# get list of proximal and distal 3'ss for stregth analysis (20nt intron + 3nt exon)
proximal_3ss_list = data_df.index.map(lambda idx: (seq_dict[idx][-28:-5]) if idx in seq_dict else None)
distal_3ss_list = data_df.index.map(lambda idx: (seq_dict[idx][-25:-2]) if idx in seq_dict else None)

#calculate maxentscan scores
data_df['proximal_3ss_score'] = maxentscan_score_3ss(proximal_3ss_list)
data_df['distal_3ss_score'] = maxentscan_score_3ss(distal_3ss_list)

data_df

,name,backbone,sequence complete,upstream exon,intron,downstream exon,NNAGNAG type,PSI ctrl. (s1),PSI ctrl. (s2),PSI ctrl. (s3),...,3end_1_base,3end_2_base,3end_3_base,3end_4_base,3end_5_base,3end_6_base,3end_7_base,3end_8_base,proximal_3ss_score,distal_3ss_score
0,ppp1r12c CAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CCAGCAG,64.04,68.57,65.51,...,c,a,g,G,A,G,C,G,5.97,10.49
1,gpank1 CAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CCAGCAG,72.31,73.20,71.15,...,c,a,g,G,A,G,C,G,6.17,8.95
2,acin1 CAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,ACAGCAG,65.57,71.78,67.65,...,c,a,g,G,A,G,C,G,2.18,-1.58
3,ankrd11 AAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,GAAGCAG,66.78,66.22,68.59,...,c,a,g,G,A,G,C,G,7.88,6.71
4,ppp1r12c CAGCAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CCAGCAG,65.04,62.39,55.11,...,c,a,g,C,T,T,G,A,5.97,9.21
5,ppp1r12c UAGUAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CTAGTAG,59.14,61.24,57.41,...,u,a,g,C,T,T,G,A,2.76,4.36
6,ppp1r12c CAGAAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CCAGAAG,86.88,83.07,81.34,...,a,a,g,C,T,T,G,A,7.20,5.06
7,ppp1r12c AAGCAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CAAGCAG,19.60,23.06,14.17,...,c,a,g,C,T,T,G,A,2.59,8.12
9,ppp1r12c UAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CTAGCAG,65.74,69.00,NaN,...,c,a,g,G,A,G,C,G,5.62,9.63
10,ppp1r12c CAGUAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CCAGTAG,80.91,78.51,NaN,...,t,a,g,G,A,G,C,G,5.75,9.16


##### 4. Get features: branchpoint

In [10]:
# Write separate FASTA files for short and long introns
write_fasta(seq_dict, "short_introns.fa", trim=(0, 8))
write_fasta(seq_dict, "long_introns.fa", trim=(0, 5))

# Run BPfinder
shortbp_df = find_bp('short_introns.fa')
longbp_df = find_bp('long_introns.fa')

# Merge information from BP if using proximal or distal sites
longbp_df['bp_same'] = 1
longbp_df['ss_dist'] = longbp_df['ss_dist']-3
longbp_df = longbp_df[['seq_id', 'ss_dist', 'bp_seq', 'bp_same']]
mergebp_df = shortbp_df.merge(longbp_df, on=['seq_id', 'ss_dist', 'bp_seq'], how='left')
mergebp_df['bp_same'] = mergebp_df['bp_same'].fillna(0)

# Split the 'bp_seq' column into 9 separate columns
mergebp_df['bp_seq'] = mergebp_df['bp_seq'].str.upper()
bpsplit_df = mergebp_df["bp_seq"].apply(lambda x: pd.Series(list(x)))
# Rename the columns for clarity
bpsplit_df.columns = [f"bp_{i+1}_base" for i in range(9)]
# Merge back with the bp DataFrame
mergebp_df = pd.concat([mergebp_df, bpsplit_df], axis=1)

# Merge BP information to the data dataframe
data_df['seq_id'] = data_df.index.tolist()
data_df = data_df.merge(mergebp_df, how="left", on="seq_id")
data_df = data_df.drop(columns=['bp_seq', 'seq_id'])

data_df

,name,backbone,sequence complete,upstream exon,intron,downstream exon,NNAGNAG type,PSI ctrl. (s1),PSI ctrl. (s2),PSI ctrl. (s3),...,bp_same,bp_1_base,bp_2_base,bp_3_base,bp_4_base,bp_5_base,bp_6_base,bp_7_base,bp_8_base,bp_9_base
0,ppp1r12c CAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CCAGCAG,64.04,68.57,65.51,...,1,T,A,C,T,G,A,C,A,T
1,gpank1 CAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CCAGCAG,72.31,73.20,71.15,...,1,T,A,C,T,G,A,C,A,T
2,acin1 CAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,ACAGCAG,65.57,71.78,67.65,...,1,T,A,C,T,G,A,C,A,T
3,ankrd11 AAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,GAAGCAG,66.78,66.22,68.59,...,1,T,A,C,T,G,A,C,A,T
4,ppp1r12c CAGCAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CCAGCAG,65.04,62.39,55.11,...,1,C,T,C,T,A,A,C,T,G
5,ppp1r12c UAGUAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CTAGTAG,59.14,61.24,57.41,...,1,C,T,C,T,A,A,C,T,G
6,ppp1r12c CAGAAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CCAGAAG,86.88,83.07,81.34,...,1,C,T,C,T,A,A,C,T,G
7,ppp1r12c AAGCAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CAAGCAG,19.60,23.06,14.17,...,1,C,T,C,T,A,A,C,T,G
8,ppp1r12c UAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CTAGCAG,65.74,69.00,NaN,...,1,T,A,C,T,G,A,C,A,T
9,ppp1r12c CAGUAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CCAGTAG,80.91,78.51,NaN,...,1,T,A,C,T,G,A,C,A,T


##### 5. Adjust and save final dataframe

In [11]:
# Sequence related features adjustment
base_cols = [col for col in data_df.columns if col.endswith('_base')]
# Capitalize bases
data_df[base_cols] = data_df[base_cols].apply(lambda col: col.str.upper())
# Change T to U
data_df[base_cols] = data_df[base_cols].replace('T', 'U')

data_df

,name,backbone,sequence complete,upstream exon,intron,downstream exon,NNAGNAG type,PSI ctrl. (s1),PSI ctrl. (s2),PSI ctrl. (s3),...,bp_same,bp_1_base,bp_2_base,bp_3_base,bp_4_base,bp_5_base,bp_6_base,bp_7_base,bp_8_base,bp_9_base
0,ppp1r12c CAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CCAGCAG,64.04,68.57,65.51,...,1,U,A,C,U,G,A,C,A,U
1,gpank1 CAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CCAGCAG,72.31,73.20,71.15,...,1,U,A,C,U,G,A,C,A,U
2,acin1 CAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,ACAGCAG,65.57,71.78,67.65,...,1,U,A,C,U,G,A,C,A,U
3,ankrd11 AAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,GAAGCAG,66.78,66.22,68.59,...,1,U,A,C,U,G,A,C,A,U
4,ppp1r12c CAGCAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CCAGCAG,65.04,62.39,55.11,...,1,C,U,C,U,A,A,C,U,G
5,ppp1r12c UAGUAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CTAGTAG,59.14,61.24,57.41,...,1,C,U,C,U,A,A,C,U,G
6,ppp1r12c CAGAAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CCAGAAG,86.88,83.07,81.34,...,1,C,U,C,U,A,A,C,U,G
7,ppp1r12c AAGCAG,pcDNA3,GACGGATCGGGAGATCTCCCGATCCCCTATGGTGCACTCTCAGTAC...,AACCACCCCCTGCAGAACCCAGAACCCTCAATGGCGTCTCCTCCCC...,gtgagtggatgccctgagaggtcctgtgaggctgggagggcccgga...,CTTGAAGAGGCCCCCTTCTCCAGGCGCTTTGGCCTCCTGAAGACAG...,CAAGCAG,19.60,23.06,14.17,...,1,C,U,C,U,A,A,C,U,G
8,ppp1r12c UAGCAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CTAGCAG,65.74,69.00,NaN,...,1,U,A,C,U,G,A,C,A,U
9,ppp1r12c CAGUAG,GFPC1,TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCA...,ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCC...,gtaagtatcaacgcgttacaagacaggtttaaggagaccaatagaa...,GAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCG...,CCAGTAG,80.91,78.51,NaN,...,1,U,A,C,U,G,A,C,A,U


In [12]:
# Save the final dataframe to a new CSV file
data_df.to_csv(f"{OUTPUT_DIR}/{OUTPUT_PREFIX}_features.csv", index=False)